# Session 09 — Pandas Cleaning, Combining and Reporting
## Python Fundamentals & Practical Applications — 48-Hour Edition

**STUDENT EDITION — WRITE YOUR OWN CODE**  
**Duration:** 4 classroom hours (including a 30-minute break)  
**Audience:** complete beginners and learners transferring from spreadsheets or other programming languages

> **Beginner promise:** We inspect imperfect synthetic data, make one responsible change at a time, combine related tables, and produce a report that can be checked.
> **Professional habit:** Never hide a data-quality decision. Record what was missing, what changed, and why.

In [ ]:
student_name = input("Enter your name: ")
print(f"{student_name}, welcome to the Pandas data-quality session.")

## Learning outcomes

By the end of this session, you should be able to:

1. Inspect and convert data types.
2. Measure missing and duplicate values.
3. Clean text categories using a documented rule.
4. Choose and apply an appropriate `fillna()` strategy.
5. Convert dates and create date summaries.
6. Stack tables with `concat()` and combine tables with every common `merge()` type.
7. Validate joins and investigate unmatched records.
8. Export a checked report.

**Total: 240 minutes.** Date analysis and exploratory analysis may be challenge work.

## Google Colab quick start — ابدأ من هنا

Open the notebook in [Google Colab](https://colab.research.google.com/) or upload it using **File → Upload notebook**. Run the setup cell before the examples.

The course datasets are synthetic and are supplied in the course `datasets` folder.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

IS_COLAB = "google.colab" in sys.modules
if IS_COLAB:
    COURSE_ROOT = Path("/content/python_48h_course")
else:
    HERE = Path.cwd()
    COURSE_ROOT = HERE

DATA_DIR = COURSE_ROOT / "datasets"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



print("Dataset folder:", DATA_DIR)
print("Output folder:", OUTPUT_DIR)

* Visit [downgit page](https://downgit.github.io/#/home) , paste the target dataset folder URL: `https://github.com/Soliman2020/python_48h_notebooks/tree/master/datasets`, then click on `Download` button.
* On your PC, after downloading the zip file, Unzip it and upload the inner files to `dataset folder` created its directory before on the previous code cell. 

In [ ]:
REQUIRED_DATA_FILES = [
    "messy_work_items.csv",
    "energy_readings.csv",
    "sector_lookup.csv",
]
missing_files = [name for name in REQUIRED_DATA_FILES if not (DATA_DIR / name).is_file()]
if missing_files:
    raise FileNotFoundError(
        f"Missing: {', '.join(missing_files)}\n"
        "Teaching datasets are missing from:\n"
        f"{DATA_DIR}.\n"
        "Add the course datasets folder/files, then run this cell again. ")
    
print("Required datasets found:", len(REQUIRED_DATA_FILES))

## Reference-data check — inspect files before cleaning

The **reference data** is the source file we are allowed to inspect and transform. Keep the original file unchanged, load it into a DataFrame, and save cleaned results to `OUTPUT_DIR`.

Before applying `fillna()`, check which files actually exist, their columns, row counts, data types, missing values, duplicate rows, and possible key columns. A lookup/reference file must also have unique values for its join key before a `many_to_one` merge.

In [ ]:
from pathlib import Path

# Inspect the files available in the reference-data folder.
reference_files = sorted(DATA_DIR.glob("*.csv"))
print("Reference CSV files:", [file.name for file in reference_files])

for file in reference_files:
    reference = pd.read_csv(file)
    print(f"\n--- {file.name} ---")
    print("Shape:", reference.shape)
    print("Columns:", reference.columns.to_list())
    print("Duplicate rows:", reference.duplicated().sum())
    print("Missing values:")
    print(reference.isna().sum())
    print("Data types:")
    print(reference.dtypes)

### Guided activity — profile one reference file

Choose one CSV file from `reference_files` and answer in comments:

1. Which columns contain missing values?
2. Which columns should be numeric, dates, categories, or join keys?
3. Which missing values could be filled, and which should remain missing or be removed?
4. Which column should be unique if the file is used as lookup data?

Do not overwrite the source CSV. Clean a copy and write it to `OUTPUT_DIR`.

In [ ]:
# Choose a file after inspecting the list above.
# reference = pd.read_csv(DATA_DIR / "your_file.csv")
# reference_clean = reference.copy()
#
# Example pattern — replace column names and strategy after inspecting meaning:
# reference_clean["category"] = reference_clean["category"].fillna("Unknown")
# reference_clean["measure"] = pd.to_numeric(
#     reference_clean["measure"], errors="coerce"
# )
# reference_clean.to_csv(OUTPUT_DIR / "reference_clean.csv", index=False)
#
# Verify the saved copy without changing the original file:
# print(reference_clean.isna().sum())
# print("Saved:", OUTPUT_DIR / "reference_clean.csv")

## 1. Inspect the mess before changing it

Cleaning is not guessing. First measure the problem. Work on a copy so the raw data remains available for comparison.

The dataset includes missing values, repeated case IDs, inconsistent text, and numeric values that need validation.

In [ ]:
raw = pd.read_csv(DATA_DIR / "messy_work_items.csv")

print("Rows and columns:", raw.shape)

In [ ]:
# Display the first few rows of the raw DataFrame to get an overview of the data
raw.head()

In [ ]:
# Display information about the DataFrame, including data types and non-null counts
raw.info()

In [ ]:
# Check for hours_open values and their counts, including NaN values
raw.hours_open.value_counts(dropna=False)

In [ ]:
print("Missing values:")
print(raw.isna().sum())

In [ ]:
# Another way to check for missing values using isnull()
raw.isnull().sum()

In [ ]:
# Check for duplicate rows in the DataFrame
raw.duplicated().sum()

In [ ]:
# Check for duplicate case IDs in the 'case_id' column
print("Duplicate case IDs:", raw["case_id"].duplicated().sum())

In [ ]:
print("Status values, including missing values:")
print(raw["status"].value_counts(dropna=False))

In [ ]:
# Keep the source CSV unchanged and work on a copy.
clean = raw.drop_duplicates(subset="case_id", keep="first").copy()

In [ ]:
# Invalid numeric text becomes missing so it can be handled deliberately.
clean["hours_open"] = pd.to_numeric(clean["hours_open"], errors="coerce")

In [ ]:
# Recheck the data types of the columns in the cleaned DataFrame
clean["hours_open"].dtype

In [ ]:
# Standardise text fields before inspecting their categories.
clean["sector"] = clean["sector"].str.strip().str.title()
clean["status"] = clean["status"].str.strip().str.lower()

print("Rows before duplicate rule:", len(raw))
print("Rows after duplicate rule:", len(clean))
print("Missing values after type/text cleaning:")
print(clean.isna().sum())

# Do not fill or drop hours_open yet. First compare possible strategies below.
clean_before_missing = clean.isna().sum()

# Example options — choose only one after considering the meaning of the field:
# clean["hours_open"] = clean["hours_open"].fillna(clean["hours_open"].median())
# clean["status"] = clean["status"].fillna("unknown")
# clean = clean.dropna(subset=["hours_open"])

print("Source file remains unchanged:", DATA_DIR / "messy_work_items.csv")

### Guided activity 1 — make a data-quality checklist

Inspect `raw.dtypes`. Identify one column that should be numeric and one that should be a date. Which columns have missing values? Write the answers in a comment before running your code.

In [ ]:
print(raw.dtypes)
# TODO: inspect one more quality measure of your choice


## 2. Clean types, text, duplicates, and missing values

A cleaning rule must match the meaning of the field:

- Strip spaces and standardise case for categories.
- Convert numeric text with `errors="coerce"` when invalid values should become missing.
- Keep the first record for a duplicate ID only when that is the documented rule.
- Do not replace missing measurements with zero unless zero really means “measured zero”.

### Choosing a `fillna()` strategy

`fillna()` is not a universal repair button. Choose the strategy from the meaning of the column and record the decision:

| Situation | Possible strategy | Warning |
|---|---|---|
| A record cannot be used without the value | `dropna(subset=[...])` | You may change the population being analysed. |
| Missing numeric value is genuinely zero | `.fillna(0)` | Do not use this merely because it is convenient. |
| Numeric values are roughly symmetric | `.fillna(column.mean())` | The mean can be distorted by outliers. |
| Numeric values are skewed or contain outliers | `.fillna(column.median())` | The filled value is an estimate, not a measurement. |
| Missing category has a meaningful label | `.fillna("Unknown")` | Keep “Unknown” separate from real categories. |
| A value should follow the previous/next observation | `.ffill()` or `.bfill()` | Only use when the row order has meaning. |

Always compare missing counts before and after filling, and keep the raw data unchanged.

In [ ]:
clean = raw.drop_duplicates(subset="case_id", keep="first").copy()

clean["sector"] = clean["sector"].str.strip().str.title()

clean["status"] = clean["status"].str.strip().str.lower()

clean["hours_open"] = pd.to_numeric(clean["hours_open"], errors="coerce")

print("Rows before:", len(raw))
print("Rows after duplicate rule:", len(clean))

print("Missing hours after conversion:", clean["hours_open"].isna().sum())

In [ ]:
# Drop rows with missing hours_open values
clean = clean.dropna(subset=["hours_open"])

In [ ]:
# Check for missing values in the cleaned DataFrame
clean.isnull().sum()

### Guided activity 2 — choose a responsible missing-value action

For `hours_open`, decide whether to keep missing values, drop rows, or fill them. Explain the decision in a comment. Apply a targeted operation only if you can justify it.

In [ ]:
# TODO: choose one strategy and explain why it fits the meaning of the column.
# Examples: dropna for unusable records, fillna(0) for a true zero,
# fillna(median) for a numeric measure, or fillna("Unknown") for a category.
# Do not use one fill rule for every column.

## 3. Dates turn records into timelines

Text that looks like a date is still text until it is parsed. Use `pd.to_datetime()` and then access components such as `.dt.year` and `.dt.month`.

A month summary can reveal changes that are difficult to see row by row.

In [ ]:
clean["created_date"] = pd.to_datetime(clean["created_date"], errors="coerce")

clean["created_month"] = clean["created_date"].dt.to_period("M")

monthly = clean.groupby("created_month").agg(cases=("case_id", "count")).reset_index()

print(monthly)

In [ ]:
lookup = pd.read_csv(DATA_DIR / "sector_lookup.csv")
lookup["sector"] = lookup["sector"].str.strip().str.title()

# Reference-data rule: the lookup key must be unique before many_to_one merge.
duplicate_lookup_keys = lookup["sector"].duplicated(keep=False)
print("Duplicate lookup keys:", lookup.loc[duplicate_lookup_keys, "sector"].unique())
if duplicate_lookup_keys.any():
    raise ValueError("sector_lookup.csv must contain one row per sector before merging.")

enriched = clean.merge(
    lookup,
    on="sector",
    how="left",
    validate="many_to_one",
    indicator=True,
)
print(enriched[["case_id", "sector", "sla_hours", "_merge"]].head())
print("Rows without a matched SLA:", (enriched["_merge"] == "left_only").sum())

# An unmatched SLA is unknown, not automatically an SLA breach.
enriched["breaches_sla"] = (
    enriched["sla_hours"].notna()
    & (enriched["hours_open"] > enriched["sla_hours"])
)

### Guided activity 3 — investigate a date field

Convert the date in `energy_readings.csv`. Print the earliest date, latest date, and number of readings per month. If the file has a different date-column name, inspect `columns` first.

In [ ]:
energy = pd.read_csv(DATA_DIR / "energy_readings.csv")
print(energy.columns.to_list())
# TODO: parse the date column and create a monthly count


## 4. Combine tables with `concat()` and `merge()`

Use the operation that matches the question:

- `concat()` stacks rows or columns; it does not match records by a key.
- `merge()` matches records using one or more key columns.

Common merge types:

- **left**: keep every row from the left table; add matches from the right.
- **right**: keep every row from the right table.
- **inner**: keep only keys found in both tables.
- **outer**: keep every key from both tables and show unmatched values as missing.

`validate=` checks the expected relationship, and `indicator=True` helps explain which rows matched. Before merging, standardise the key in both tables.

In [ ]:
lookup = pd.read_csv(DATA_DIR / "sector_lookup.csv")
lookup["sector"] = lookup["sector"].str.strip().str.title()
enriched = clean.merge(lookup, on="sector", how="left", validate="many_to_one")
print(enriched[["case_id", "sector", "sla_hours"]].head())
print("Rows without a matched SLA:", enriched["sla_hours"].isna().sum())
enriched["breaches_sla"] = enriched["hours_open"] > enriched["sla_hours"]

### `concat()` versus `merge()`

Use `concat()` to stack tables with the same kind of columns. Use `merge()` to add columns by matching keys. Confusing these operations can silently produce the wrong report.

### Demonstration — compare merge types

The examples use small in-memory tables so the row-preservation rule is visible. `indicator=True` labels each result row as `left_only`, `right_only`, or `both`.

In [ ]:
left_demo = pd.DataFrame({"key": [1, 2, 3], "left_value": ["A", "B", "C"]})
right_demo = pd.DataFrame({"key": [2, 3, 4], "right_value": ["X", "Y", "Z"]})

for join_type in ["left", "right", "inner", "outer"]:
    result = left_demo.merge(
        right_demo,
        on="key",
        how=join_type,
        indicator=True,
    )
    print(f"\n{join_type} merge ({len(result)} rows)")
    print(result)

### Guided activity 4 — check the join

Compare the row count before and after the merge. Investigate any rows with missing `sla_hours`. What does an unmatched key mean for the report?

Then repeat the merge with `indicator=True` and inspect the `_merge` counts. Which join type would you choose if the report must preserve every work item? Explain why.

In [ ]:
print("Rows before merge:", len(clean))
print("Rows after merge:", len(enriched))
print(enriched.loc[enriched["sla_hours"].isna(), ["case_id", "sector"]])

# TODO: repeat the merge with indicator=True and summarise _merge values.
# TODO: explain why a left merge is appropriate for the SLA report.

## Practice tasks

Use comments to document every cleaning and combining decision. Check row counts after each operation.

### Applied task A — `S09-E01`
Find missing values and duplicate IDs in `messy_work_items.csv`. Report the counts before changing the data. Then write one sentence explaining which missing-value strategy you would use for one column and why.

In [ ]:
# Your code here

### Applied task B — `S09-E02`
Clean the status column and create a table of status counts.


In [ ]:
# Your code here
# Hint: strip and lowercase first, then choose a documented fillna()
# strategy for missing statuses before calling value_counts(dropna=False).

### Applied task C — `S09-E03`
Prepare the work-items table and `sector_lookup.csv` for joining. Demonstrate `left`, `inner`, `outer`, and `right` merges with `indicator=True`. Compare their row counts and explain which merge is appropriate when every work item must remain in the report.

In [ ]:
# Your code here
# Hint: loop over ["left", "inner", "outer", "right"] and print
# len(result) plus result["_merge"].value_counts().to_dict().

### Challenge — `S09-E04`
Create a monthly report with total cases and average hours open. Export it as CSV. Include a short note stating how you handled missing dates and missing `hours_open` values.

In [ ]:
# Your code here

## Guided mini-project — Data-quality and SLA report

Build a report from `messy_work_items.csv` and `sector_lookup.csv`. Your report must:

1. Show row counts before and after the duplicate rule.
2. Show missing-value counts before cleaning and after your chosen `fillna()`/`dropna()` rules.
3. Standardise the join key and perform a validated left merge with `indicator=True`.
4. Explain why a left merge is appropriate and briefly compare it with an inner merge.
5. Flag SLA breaches without treating an unmatched SLA as a breach.
6. Export a summary by sector to `session_09_sla_report.csv`.
7. Explain one assumption and one limitation.

Keep the raw table unchanged and make your decisions visible in code comments.

In [ ]:
# TODO: complete the report workflow and export the summary


## Optional EDA extension

Use `describe()`, `value_counts()`, or one simple chart to investigate a question about the cleaned data. A chart should support a question; it is not a replacement for checking the table.

In [ ]:
# Optional: investigate one question about the cleaned or enriched data.


## Exit check — answer from memory

1. Why inspect missing values before cleaning?
2. Which `fillna()` strategy would fit a missing category, and what label might you use?
3. Why can `fillna(0)` be a bad decision for a measurement?
4. What is the difference between `concat()` and `merge()`?
5. How do `left`, `inner`, `outer`, and `right` merges differ?
6. What does a cross merge produce, and why can it create unexpectedly many rows?
7. What does `pd.to_datetime()` do?
8. What does a left merge preserve?
9. Why are `validate="many_to_one"` and `indicator=True` useful?

## Common mistakes and next steps

- Do not clean before measuring the original problem.
- Keep a raw copy when making transformations.
- Choose `fillna()` based on the meaning of the field, not convenience.
- Normalise join keys before merging.
- Check unmatched keys after a merge, especially with `indicator=True`.
- Use `validate=` to detect unexpected duplicate keys.
- Treat missing measurements as a meaning question, not merely a coding problem.
- A correct report includes assumptions and limitations.

**Next step:** use the Pandas workflow as a foundation for charts, automation, APIs, and the capstone project.